# 02 · Propensity model

Milestone 2. Engine 1 predicts **P_i = probability that customer *i* subscribes if contacted**, using only pre-campaign information (leakage policy D-004).

- **Primary: Logistic Regression.** One-hot + scaling, `C` tuned on PR-AUC with 5-fold CV, no class weighting, then **cross-validated isotonic calibration** (D-027).
- **Challenger: Random Forest**, isotonic-calibrated, used as a benchmark.

Discrimination and calibration are measured on the full held-out test set (13,564). Precision@K is measured on the 10,000-customer campaign universe (a subset of the test set).

In [1]:
import json
import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from pacopt import config as cfg, features as ft, model as m, charts as ch

pio.renderers.default = 'notebook_connected'
pd.set_option('display.width', 140)
train, test, universe = m.load_processed()
print(f'train {train.shape}  test {test.shape}  universe {universe.shape}')
ft.assert_no_leakage(ft.MODEL_FEATURES)
print('Features:', ft.MODEL_FEATURES)

train (31647, 31)  test (13564, 31)  universe (10000, 31)
Features: ['age', 'balance_slog', 'frequency', 'never_contacted', 'prior_success', 'housing', 'loan', 'default', 'job', 'marital', 'education', 'contact', 'poutcome', 'age_band', 'recency_bucket']


## 1. Train both models

In [2]:
%%time
res = m.train_and_evaluate(train, test, universe)
print('LR best C =', res.lr_search.best_params_['clf__C'], '| CV PR-AUC =', round(res.lr_search.best_score_, 4))

LR best C = 0.3 | CV PR-AUC = 0.3742
CPU times: total: 1min 8s
Wall time: 22.7 s


In [3]:
cv = pd.DataFrame(res.lr_search.cv_results_)[['param_clf__C', 'mean_test_score', 'std_test_score']]
cv.columns = ['C', 'cv_pr_auc', 'std']
cv.round(4)

,C,cv_pr_auc,std
0,0.003,0.3688,0.0062
1,0.010,0.3720,0.0069
2,0.030,0.3735,0.0083
3,0.100,0.3741,0.0094
4,0.300,0.3742,0.0102
5,1.000,0.3741,0.0105
6,3.000,0.3738,0.0107
7,10.000,0.3739,0.0107


## 2. Evaluation: ROC-AUC, PR-AUC, Brier, calibration, precision@K

In [4]:
rows = []
for ev in (res.eval_lr, res.eval_rf):
    r = {'model': ev.name, 'ROC-AUC': ev.roc_auc, 'PR-AUC': ev.pr_auc, 'Brier': ev.brier,
         'max calibration gap (pp)': ev.max_calibration_gap_pp}
    r.update({f'precision@{k}': v for k, v in ev.precision_at_k.items()})
    rows.append(r)
summary = pd.DataFrame(rows).set_index('model').T
summary.loc['budget-implied K'] = [res.eval_lr.budget_k, res.eval_rf.budget_k]
summary.loc['base rate'] = res.eval_lr.base_rate
summary.round(4)

model,Logistic Regression,Random Forest (calibrated)
ROC-AUC,0.7526,0.7558
PR-AUC,0.3802,0.3962
Brier,0.0875,0.0865
max calibration gap (pp),1.2205,1.9844
precision@K=500,0.5680,0.5840
precision@K=1000,0.4280,0.4480
precision@K=2000,0.3015,0.3015
precision@K=budget,0.1990,0.2047
budget-implied K,4261.0000,4246.0000
base rate,0.1170,0.1170


The budget-implied K is the number of contacts the £50K budget buys when customers are contacted in descending propensity at fully-loaded channel costs (£12 mobile, £10 landline, £2 digital).

In [5]:
p_lr, p_rf = m.predict(res.lr, test), m.predict(res.rf, test)
y = test['y'].to_numpy()
c_lr, c_rf = m.curves(y, p_lr), m.curves(y, p_rf)

fig = go.Figure()
fig.add_scatter(x=c_lr['roc'].fpr, y=c_lr['roc'].tpr, name=f'Logistic Regression (AUC {res.eval_lr.roc_auc:.3f})',
                line=dict(color=ch.BLUE, width=2))
fig.add_scatter(x=c_rf['roc'].fpr, y=c_rf['roc'].tpr, name=f'Random Forest (AUC {res.eval_rf.roc_auc:.3f})',
                line=dict(color=ch.ORANGE, width=2))
ch.reference_line(fig, 0, 0, 1, 1, 'random')
ch.style(fig, 'ROC curve: held-out test set', height=430)
fig.update_xaxes(title='False positive rate', range=[0, 1])
fig.update_yaxes(title='True positive rate', range=[0, 1])
fig.show()

In [6]:
fig = go.Figure()
fig.add_scatter(x=c_lr['pr'].recall, y=c_lr['pr'].precision, name=f'Logistic Regression (AP {res.eval_lr.pr_auc:.3f})',
                line=dict(color=ch.BLUE, width=2))
fig.add_scatter(x=c_rf['pr'].recall, y=c_rf['pr'].precision, name=f'Random Forest (AP {res.eval_rf.pr_auc:.3f})',
                line=dict(color=ch.ORANGE, width=2))
base = res.eval_lr.base_rate
ch.reference_line(fig, 0, base, 1, base, f'base rate {base:.1%}', label_at='start')
ch.style(fig, 'Precision-recall curve: held-out test set', height=430)
fig.update_xaxes(title='Recall', range=[0, 1])
fig.update_yaxes(title='Precision', range=[0, 1], tickformat='.0%')
fig.show()

### Calibration
P_i is multiplied by money in EP_i = P_i × V_i − C_i, so predicted probabilities must match observed rates. The acceptance criterion is a gap of at most 1.5 pp in every decile.

In [7]:
cal = pd.concat({'LR': m.calibration_table(y, p_lr), 'RF': m.calibration_table(y, p_rf)}, axis=1)
fig = go.Figure()
for name, color, t in (('Logistic Regression', ch.BLUE, cal['LR']), ('Random Forest', ch.ORANGE, cal['RF'])):
    fig.add_scatter(x=t['predicted'], y=t['actual'], name=name, mode='lines+markers',
                    line=dict(color=color, width=2), marker=dict(size=8, line=dict(color=ch.SURFACE, width=2)),
                    hovertemplate='predicted %{x:.1%}<br>actual %{y:.1%}<extra>' + name + '</extra>')
ch.reference_line(fig, 0, 0, 0.5, 0.5, 'perfect calibration')
ch.style(fig, 'Calibration by decile: held-out test set', height=430)
fig.update_xaxes(title='Mean predicted probability', tickformat='.0%', range=[0, 0.5])
fig.update_yaxes(title='Observed subscription rate', tickformat='.0%', range=[0, 0.5])
fig.show()
cal.xs('gap_pp', axis=1, level=1).round(2).T

bin,1,2,3,4,5,6,7,8,9,10
LR,-0.68,0.31,-0.24,1.22,-0.25,-0.86,-0.65,0.74,0.71,-0.80
RF,-1.01,0.26,-0.34,0.82,0.18,-0.83,0.96,-0.11,1.64,-1.98


### Why the LR is calibrated (D-027)
Uncalibrated, the LR under-predicts its top decile by about 4 pp (predicted 39.8% vs actual 43.8%). That decile is dominated by customers whose previous campaign succeeded, where recency effects interact in a way a linear model can't fully express. Isotonic calibration is monotone, so it **doesn't change the ranking or the meaning of the coefficients**. It only re-maps the scores to the observed rates.

In [8]:
raw_lr = res.lr_search.best_estimator_
p_raw = m.predict(raw_lr, test)
pd.DataFrame({'uncalibrated gap (pp)': m.calibration_table(y, p_raw)['gap_pp'],
              'calibrated gap (pp)': m.calibration_table(y, p_lr)['gap_pp']}).round(2).T

bin,1,2,3,4,5,6,7,8,9,10
uncalibrated gap (pp),-0.81,-0.54,-0.59,1.63,0.50,1.00,0.76,0.88,0.71,-3.96
calibrated gap (pp),-0.68,0.31,-0.24,1.22,-0.25,-0.86,-0.65,0.74,0.71,-0.80


## 3. Lift and gains: what targeting by propensity buys

In [9]:
g = m.gains_table(y, p_lr)
fig = go.Figure(go.Bar(x=g.index.astype(str), y=g['rate'], marker_color=ch.BAR_BLUE,
                       marker_cornerradius=4, text=[f'{v:.0%}' for v in g['rate']],
                       textposition='outside', textfont=dict(color=ch.INK_2),
                       hovertemplate='decile %{x}<br>conversion %{y:.1%}<extra></extra>'))
ch.reference_line(fig, -0.5, base, 9.5, base, f'base rate {base:.1%}')
ch.style(fig, 'Observed conversion by LR propensity decile (1 = highest)', height=400, showlegend=False)
fig.update_xaxes(title='Propensity decile')
fig.update_yaxes(title='Conversion rate', tickformat='.0%')
fig.show()
g.round(3)

,n,positives,avg_p,rate,lift,cum_capture,cum_lift
decile,,,,,,,
1,1357,593,0.429,0.437,3.735,0.374,3.735
2,1356,231,0.177,0.170,1.456,0.519,2.596
3,1356,170,0.133,0.125,1.072,0.626,2.088
4,1357,135,0.093,0.099,0.850,0.711,1.778
5,1356,122,0.081,0.090,0.769,0.788,1.577
6,1356,103,0.073,0.076,0.649,0.853,1.422
7,1357,63,0.059,0.046,0.397,0.893,1.276
8,1356,73,0.051,0.054,0.460,0.939,1.174
9,1356,52,0.041,0.038,0.328,0.972,1.080


## 4. Explainability: top drivers

In [10]:
coef = m.lr_coefficients(res.lr_search)
top = coef.head(16).iloc[::-1]
fig = go.Figure(go.Bar(x=top['coef'], y=top['feature'], orientation='h',
                       marker_color=[ch.BLUE if c > 0 else ch.RED for c in top['coef']],
                       marker_cornerradius=3,
                       hovertemplate='%{y}<br>coef %{x:.3f}<extra></extra>'))
ch.style(fig, 'Logistic Regression: 16 largest coefficients (blue raises, red lowers propensity)',
         height=520, showlegend=False)
fig.update_xaxes(title='Coefficient (log-odds; numeric features per 1 SD)')
fig.show()
coef.head(16)[['feature', 'coef', 'odds_ratio']].round(3)

,feature,coef,odds_ratio
0,contact_unknown,-0.728,0.483
1,age_band_45-59,-0.704,0.495
2,age_band_75+,0.696,2.006
3,age_band_30-44,-0.635,0.530
4,recency_bucket_>365d,0.571,1.771
5,recency_bucket_181-365d,-0.411,0.663
6,job_housemaid,-0.401,0.669
7,recency_bucket_never,-0.387,0.679
8,marital_married,-0.358,0.699
9,prior_success,0.342,1.408


Permutation importance measures the drop in PR-AUC when each **original** feature is shuffled. It works on both models, so they can be compared directly.

In [11]:
%%time
pi = pd.concat({'LR': m.permutation_drivers(res.lr, test).set_index('feature')['importance'],
                'RF': m.permutation_drivers(res.rf, test).set_index('feature')['importance']}, axis=1)
pi = pi.sort_values('LR')
fig = go.Figure()
fig.add_bar(y=pi.index, x=pi['LR'], name='Logistic Regression', orientation='h', marker_color=ch.BLUE, marker_cornerradius=3)
fig.add_bar(y=pi.index, x=pi['RF'], name='Random Forest', orientation='h', marker_color=ch.ORANGE, marker_cornerradius=3)
ch.style(fig, 'Permutation importance (drop in PR-AUC when shuffled)', height=520)
fig.update_layout(barmode='group', bargap=0.25, bargroupgap=0.1)
fig.update_xaxes(title='Drop in PR-AUC')
fig.show()
pi.iloc[::-1].round(4)

CPU times: total: 3min 22s
Wall time: 1min 43s


,LR,RF
feature,,
prior_success,0.1314,0.0240
contact,0.0313,0.0248
age_band,0.0214,0.0020
housing,0.0178,0.0252
balance_slog,0.0143,0.0087
loan,0.0083,0.0041
recency_bucket,0.0082,0.0090
poutcome,0.0061,0.0385
frequency,0.0022,0.0057


## 5. Model choice summary

In [12]:
lr_e, rf_e = res.eval_lr, res.eval_rf
choice = pd.DataFrame({
    'Logistic Regression': [lr_e.roc_auc, lr_e.pr_auc, lr_e.brier, lr_e.max_calibration_gap_pp,
                            lr_e.precision_at_k['K=1000'], 'coefficients + odds ratios', 'seconds'],
    'Random Forest': [rf_e.roc_auc, rf_e.pr_auc, rf_e.brier, rf_e.max_calibration_gap_pp,
                      rf_e.precision_at_k['K=1000'], 'permutation importance only', 'tens of seconds'],
}, index=['ROC-AUC', 'PR-AUC', 'Brier', 'max calibration gap (pp)', 'precision@1000',
          'explainability', 'training time'])
print('Recommended:', res.recommended.upper(), '-', res.reason)
print('Selected in config.MODEL.selected:', cfg.MODEL.selected.upper())
choice

Recommended: LR - RF PR-AUC gain +0.016 is below the 0.02 switch threshold; LR kept for calibration and explainability
Selected in config.MODEL.selected: LR


,Logistic Regression,Random Forest
ROC-AUC,0.752621,0.755752
PR-AUC,0.380208,0.3962
Brier,0.087482,0.086545
max calibration gap (pp),1.22048,1.98436
precision@1000,0.428,0.448
explainability,coefficients + odds ratios,permutation importance only
training time,seconds,tens of seconds


**Decision:** keep **Logistic Regression** (calibrated) as the scoring model. The RF gains about 0.016 PR-AUC and 0.003 ROC-AUC, which is below the pre-agreed 0.02 PR-AUC switch threshold. The LR is now *better* calibrated (max gap 1.2 pp vs 2.0 pp), explainable through coefficients, and fast enough for interactive what-if re-runs. To switch, set `MODEL.selected = 'rf'` in `config.py`.

### Tail guard: propensity cap (D-028)
Isotonic calibration is a step function. In the extreme tail it is fitted on very few customers and can give scores like 98%. The table checks this on the test set. The propensity passed to the profit and optimisation engines is capped at `MODEL.propensity_cap` (90%). The raw calibrated score is kept in `p_lr`.

In [13]:
d = pd.DataFrame({'p': p_lr, 'y': y})
bands = pd.cut(d['p'], [0, 0.5, 0.7, 0.9, 1.0], include_lowest=True)
d.groupby(bands, observed=True).agg(customers=('y', 'size'), mean_predicted=('p', 'mean'),
                                    actual_rate=('y', 'mean')).round(3)

,customers,mean_predicted,actual_rate
p,,,
"(-0.001, 0.5]",13153,0.100,0.099
"(0.5, 0.7]",317,0.627,0.672
"(0.7, 0.9]",86,0.742,0.744
"(0.9, 1.0]",8,0.973,0.625


## 6. Score the campaign universe and save artifacts

In [14]:
m.save(res)
s = res.scored
print(s.shape, '| mean propensity', round(s['propensity'].mean(), 4), '| actual rate', round(universe['y'].mean(), 4))
s.sort_values('propensity_rank').head(10)

(10000, 6) | mean propensity 0.1163 | actual rate 0.117


,customer_id,p_lr,p_rf,propensity,propensity_rank,propensity_decile
9575,C143424,0.980000,0.691832,0.900000,1,1
9638,C143676,0.980000,0.739848,0.900000,2,1
9758,C144152,0.980000,0.669188,0.900000,3,1
9806,C144377,0.980000,0.711669,0.900000,4,1
9816,C144421,0.980000,0.694356,0.900000,5,1
9911,C144807,0.980000,0.669188,0.900000,6,1
9741,C144088,0.949067,0.696543,0.900000,7,1
8951,C140555,0.880000,0.691832,0.880000,8,1
9633,C143643,0.880000,0.694356,0.880000,9,1
9422,C142754,0.864545,0.735601,0.864545,10,1


In [15]:
print(json.dumps(json.loads(m.METRICS_PATH.read_text()), indent=1)[:1500])

{
 "selected": "lr",
 "recommended": "lr",
 "reason": "RF PR-AUC gain +0.016 is below the 0.02 switch threshold; LR kept for calibration and explainability",
 "lr_best_C": 0.3,
 "lr_cv_pr_auc": 0.3741660639098466,
 "lr_calibration": "isotonic",
 "lr": {
  "name": "Logistic Regression",
  "roc_auc": 0.7526211891422434,
  "pr_auc": 0.38020755315699273,
  "brier": 0.0874818857280798,
  "log_loss": 0.3066102503083004,
  "base_rate": 0.1170008846947803,
  "max_calibration_gap_pp": 1.2204804346213467,
  "precision_at_k": {
   "K=500": 0.568,
   "K=1000": 0.428,
   "K=2000": 0.3015,
   "K=budget": 0.19901431588828913
  },
  "budget_k": 4261
 },
 "rf": {
  "name": "Random Forest (calibrated)",
  "roc_auc": 0.7557519534789927,
  "pr_auc": 0.3962004669988612,
  "brier": 0.08654538485944414,
  "log_loss": 0.3039435147210114,
  "base_rate": 0.1170008846947803,
  "max_calibration_gap_pp": 1.984359790660356,
  "precision_at_k": {
   "K=500": 0.584,
   "K=1000": 0.448,
   "K=2000": 0.3015,
   "K=budg

**Milestone 2 output:** `artifacts/model.joblib` (calibrated LR, explainable LR, calibrated RF), `artifacts/metrics.json`, and `data/processed/universe_scored.parquet` with a propensity, rank and decile for each of the 10,000 customers.